In [0]:
# Bounded, side-effect-free blob decoder shared by the updated text pipeline.

import hashlib
from typing import Optional


# Version bumps intentionally invalidate older extraction results for replay.
DECOMPRESSOR_VERSION = 3
PARSER_VERSION = 3
POST_PROCESSOR_VERSION = 4

MAX_COMPRESSED_BYTES = 16 * 1024 * 1024
MAX_DECOMPRESSED_BYTES = 16 * 1024 * 1024
MAX_TEXT_CHARS = 10_000_000

OCR_MAX_PAGES = 10
OCR_MAX_PDF_SIZE_MB = 16
OCR_LANG = "eng"
OCR_PAGE_TIMEOUT_SEC = 20
PROCESS_TIMEOUT_SEC = 45

OCF_MARKER = b"ocf_blob\0"
PDF_MAGIC = b"%PDF-"
RTF_MAGIC = b"{\\rtf"
RTF_MAGIC_LOOSE = b"{\\"
ZIP_MAGIC = b"\x50\x4b\x03\x04"
OLE_MAGIC = b"\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1"
TIFF_LE_MAGIC = b"II*\x00"
TIFF_BE_MAGIC = b"MM\x00*"
JPEG_MAGIC = b"\xff\xd8\xff"
PNG_MAGIC = b"\x89PNG\r\n\x1a\n"
GIF87_MAGIC = b"GIF87a"
GIF89_MAGIC = b"GIF89a"

BLOB_LENGTH_TOLERANCE = 0.01
LZW_LOOP_BUDGET_MULT = 16


class CernerLZWError(Exception):
    pass


class OutputLimitError(CernerLZWError):
    pass


def raw_content_sha256(chunks: list) -> str:
    """Hash original chunk bytes in deterministic sequence order."""
    h = hashlib.sha256()
    for chunk in sorted(chunks or [], key=lambda x: (x["BLOB_SEQ_NUM"] is None, x["BLOB_SEQ_NUM"] or 0)):
        contents = chunk["BLOB_CONTENTS"]
        if contents is not None:
            h.update(bytes(contents))
    return h.hexdigest()


def combine_blob_chunks(chunks: list) -> bytes:
    """Combine chunks while removing only a marker at each chunk boundary.

    Removing every marker occurrence can corrupt legitimate payload bytes. Cerner's
    wrapper marker is expected at a chunk boundary, so strip only a trailing marker
    from each original chunk before concatenation.
    """
    parts = []
    for chunk in sorted(chunks or [], key=lambda x: (x["BLOB_SEQ_NUM"] is None, x["BLOB_SEQ_NUM"] or 0)):
        value = chunk["BLOB_CONTENTS"]
        if value is None:
            continue
        value = bytes(value)
        if value.endswith(OCF_MARKER):
            value = value[:-len(OCF_MARKER)]
        parts.append(value)
    return b"".join(parts)


def strip_trailing_ocf_marker(data: bytes) -> bytes:
    if data and data.endswith(OCF_MARKER):
        return data[:-len(OCF_MARKER)]
    return data


def sniff_container_magic(data: bytes) -> bool:
    if not data:
        return False
    head = data[:16]
    return any(head.startswith(m) for m in (
        PDF_MAGIC, RTF_MAGIC, ZIP_MAGIC, OLE_MAGIC, TIFF_LE_MAGIC,
        TIFF_BE_MAGIC, JPEG_MAGIC, PNG_MAGIC, GIF87_MAGIC, GIF89_MAGIC,
    ))


def _bounded_extend(output: bytearray, value: bytes, max_output_bytes: int) -> None:
    if len(output) + len(value) > max_output_bytes:
        raise OutputLimitError(f"decompressed output exceeds {max_output_bytes} bytes")
    output.extend(value)


def cerner_lzw_decompress(
    data: bytes,
    *,
    loop_budget: Optional[int] = None,
    max_output_bytes: int = MAX_DECOMPRESSED_BYTES,
) -> bytes:
    """MSB-first Cerner OCF LZW with an enforced output limit."""
    if not data:
        return b""

    clear_code, eoi_code, first_code = 256, 257, 258
    max_code = (1 << 13) - 1
    dictionary = {i: bytes([i]) for i in range(256)}
    next_code, code_size = first_code, 9
    output = bytearray()
    previous = b""
    bit_buffer = bits_in_buffer = byte_pos = 0

    def read_code():
        nonlocal bit_buffer, bits_in_buffer, byte_pos
        while bits_in_buffer < code_size and byte_pos < len(data):
            bit_buffer = (bit_buffer << 8) | data[byte_pos]
            bits_in_buffer += 8
            byte_pos += 1
        if bits_in_buffer < code_size:
            return None
        code = (bit_buffer >> (bits_in_buffer - code_size)) & ((1 << code_size) - 1)
        bits_in_buffer -= code_size
        bit_buffer &= (1 << bits_in_buffer) - 1
        return code

    if loop_budget is None:
        loop_budget = max(1024, len(data) * LZW_LOOP_BUDGET_MULT)

    code = read_code()
    if code is None:
        return b""
    if code == clear_code:
        code = read_code()
        if code is None or code == eoi_code:
            return b""
    if code >= 256:
        raise CernerLZWError(f"invalid first code {code}")
    _bounded_extend(output, bytes([code]), max_output_bytes)
    previous = bytes([code])

    for iteration in range(loop_budget):
        code = read_code()
        if code is None or code == eoi_code:
            break
        if code == clear_code:
            dictionary = {i: bytes([i]) for i in range(256)}
            next_code, code_size, previous = first_code, 9, b""
            code = read_code()
            if code is None or code == eoi_code:
                break
            if code >= 256:
                raise CernerLZWError(f"invalid post-clear code {code}")
            current = bytes([code])
            _bounded_extend(output, current, max_output_bytes)
            previous = current
            continue

        if code < 256:
            current = bytes([code])
        elif code in dictionary:
            current = dictionary[code]
        elif code == next_code and previous:
            current = previous + previous[:1]
        else:
            raise CernerLZWError(
                f"unknown code {code} at iteration {iteration} (next={next_code})"
            )

        _bounded_extend(output, current, max_output_bytes)
        if previous and next_code <= max_code:
            dictionary[next_code] = previous + current[:1]
            next_code += 1
            if next_code == 511 and code_size == 9:
                code_size = 10
            elif next_code == 1023 and code_size == 10:
                code_size = 11
            elif next_code == 2047 and code_size == 11:
                code_size = 12
            elif next_code == 4095 and code_size == 12:
                code_size = 13
        previous = current
    else:
        raise CernerLZWError(f"loop budget {loop_budget} exceeded")

    return bytes(output)


def tiff_lzw_decompress(data: bytes, max_output_bytes: int = MAX_DECOMPRESSED_BYTES) -> bytes:
    """MSB-first TIFF-style LZW fallback with bounded output."""
    if not data:
        return b""
    clear_code, eoi_code, first_code, max_dict = 256, 257, 258, 8192
    dictionary = {i: bytes([i]) for i in range(256)}
    next_code, code_size = first_code, 9
    output, previous = bytearray(), b""
    bit_buffer = bits_in_buffer = byte_pos = 0

    def read_code():
        nonlocal bit_buffer, bits_in_buffer, byte_pos
        while bits_in_buffer < code_size and byte_pos < len(data):
            bit_buffer = (bit_buffer << 8) | data[byte_pos]
            bits_in_buffer += 8
            byte_pos += 1
        if bits_in_buffer < code_size:
            return None
        value = (bit_buffer >> (bits_in_buffer - code_size)) & ((1 << code_size) - 1)
        bits_in_buffer -= code_size
        bit_buffer &= (1 << bits_in_buffer) - 1
        return value

    for _ in range(max(1024, len(data) * 10)):
        code = read_code()
        if code is None or code == eoi_code:
            break
        if code == clear_code:
            dictionary = {i: bytes([i]) for i in range(256)}
            next_code, code_size, previous = first_code, 9, b""
            continue
        if code < 256:
            current = bytes([code])
        elif code in dictionary:
            current = dictionary[code]
        elif code == next_code and previous:
            current = previous + previous[:1]
        else:
            raise CernerLZWError(f"invalid TIFF-LZW code {code}")
        _bounded_extend(output, current, max_output_bytes)
        if previous and next_code < max_dict:
            dictionary[next_code] = previous + current[:1]
            next_code += 1
            if next_code == 511 and code_size == 9:
                code_size = 10
            elif next_code == 1023 and code_size == 10:
                code_size = 11
            elif next_code == 2047 and code_size == 11:
                code_size = 12
            elif next_code == 4095 and code_size == 12:
                code_size = 13
        previous = current
    return bytes(output)


def tolerant_lzw_decompress(data: bytes, max_output_bytes: int = MAX_DECOMPRESSED_BYTES) -> bytes:
    """LSB-first fallback. Invalid codes fail instead of fabricating output."""
    if not data:
        return b""
    clear_code, end_code, first_code, max_code = 256, 257, 258, 65535
    dictionary = {i: bytes([i]) for i in range(256)}
    next_code, code_size, max_for_size = first_code, 9, 512
    output, previous = bytearray(), b""
    bit_buffer = bits_in_buffer = byte_pos = 0

    for _ in range(max(1024, len(data) * 8)):
        while bits_in_buffer < code_size and byte_pos < len(data):
            bit_buffer |= data[byte_pos] << bits_in_buffer
            bits_in_buffer += 8
            byte_pos += 1
        if bits_in_buffer < code_size:
            break
        code = bit_buffer & ((1 << code_size) - 1)
        bit_buffer >>= code_size
        bits_in_buffer -= code_size
        if code == end_code:
            break
        if code == clear_code:
            dictionary = {i: bytes([i]) for i in range(256)}
            next_code, code_size, max_for_size, previous = first_code, 9, 512, b""
            continue
        if code < 256:
            current = bytes([code])
        elif code in dictionary:
            current = dictionary[code]
        elif code == next_code and previous:
            current = previous + previous[:1]
        else:
            raise CernerLZWError(f"invalid tolerant-LZW code {code}")
        _bounded_extend(output, current, max_output_bytes)
        if previous and next_code <= max_code:
            dictionary[next_code] = previous + current[:1]
            next_code += 1
            if next_code >= max_for_size and code_size < 16:
                code_size += 1
                max_for_size *= 2
        previous = current
    return bytes(output)


def _set_length_metrics(value: bytes, expected_length: Optional[int], metrics: dict) -> None:
    metrics["decompressed_bytes"] = len(value)
    metrics["declared_blob_length"] = expected_length
    if expected_length and expected_length > 0:
        metrics["blob_length_mismatch"] = (
            abs(len(value) - int(expected_length)) / float(expected_length)
            > BLOB_LENGTH_TOLERANCE
        )
    else:
        metrics["blob_length_mismatch"] = False


def decompress(
    raw: bytes,
    compression_cd,
    chunk_count: int,
    blob_length: Optional[int],
    metrics: dict,
    max_output_bytes: int = MAX_DECOMPRESSED_BYTES,
):
    """Return ``(bytes, error)`` and never intentionally exceed output limit."""
    if not raw:
        return None, "Empty content"
    metrics["compressed_bytes"] = len(raw)
    metrics["chunk_count"] = int(chunk_count or 0)
    if len(raw) > MAX_COMPRESSED_BYTES:
        return None, f"Compressed too large: {len(raw)} bytes"

    try:
        code = int(compression_cd) if compression_cd is not None else None
    except Exception:
        code = None

    cleaned = strip_trailing_ocf_marker(raw)
    if code == 727 or (code == 728 and sniff_container_magic(cleaned)):
        if len(cleaned) > max_output_bytes:
            return None, f"Decompressed too large: {len(cleaned)} bytes"
        metrics["decompression_strategy"] = "uncompressed" if code == 727 else "uncompressed_sniffed"
        _set_length_metrics(cleaned, blob_length, metrics)
        return cleaned, None
    if code != 728:
        return None, f"Unknown compression: {compression_cd}"

    errors = []
    for name, decoder in (
        ("cerner_lzw", cerner_lzw_decompress),
        ("tiff_lzw", tiff_lzw_decompress),
        ("tolerant_lzw", tolerant_lzw_decompress),
    ):
        try:
            result = decoder(cleaned, max_output_bytes=max_output_bytes)
            if result:
                metrics["decompression_strategy"] = name
                _set_length_metrics(result, blob_length, metrics)
                return result, None
        except OutputLimitError as exc:
            return None, f"Decompressed too large: {exc}"
        except Exception as exc:
            errors.append(f"{name}:{type(exc).__name__}:{str(exc)[:160]}")

    return None, "LZW failed all bounded decoders: " + " | ".join(errors[-3:])


def classify_file_type(data: bytes):
    if not data:
        return None
    head = data[:16]
    if head.startswith(PDF_MAGIC):
        return ("pdf", "pdf", "application/pdf")
    if head.startswith(JPEG_MAGIC):
        return ("jpg", "jpeg", "image/jpeg")
    if head.startswith(PNG_MAGIC):
        return ("png", "png", "image/png")
    if head.startswith(TIFF_LE_MAGIC) or head.startswith(TIFF_BE_MAGIC):
        return ("tif", "tiff", "image/tiff")
    return None


print(
    f"updated blob_decoder loaded: decompressor={DECOMPRESSOR_VERSION}, "
    f"compressed_limit={MAX_COMPRESSED_BYTES}, decompressed_limit={MAX_DECOMPRESSED_BYTES}"
)